# Graph Model and Scikit-Learn
There may be times that typical graph models may not produce sufficiently performing models. One culprit is the size of the dataset is too small to be effectively utilized by deep learning methods. To combat this, graph features can be utilized as the feature vector as input for traditional machine learning (ML) models from Scikit-Learn. There have been several examples of this by researchers ([here](https://www.nature.com/articles/s42256-024-00923-6), [here](https://github.com/Chokyotager/NotYetAnotherNightshade/tree/main), and [here](https://jonswain.github.io/ai/cheminformatics/data%20science/machine%20learning/2025/09/12/chemprop-rf.html
)). To do this, the molecules will be converted into molecular graphs. Then the graph latent space will be exported. These features are in numpy format, which can then be used to build traditional ML models.

This tutorial will demo how to create molecular graph features and output the latent space. This will then be used to build a RandomForrest classifier model and predict sample molecules.

## Import packages
The MolMiner comes with four graph models - Graph Convolutional Neural Network (GCN), Graph Isomorphism Network (GINE), Graph Sage (GSAGE), and Graph Attention Networks (GAT). These classes can be imported from the **latent_space** module.

In [1]:
import pandas as pd
from tqdm import tqdm
from mminer_graphs import MolToGraph
from mminer_graphs.model.latent_space import GCN

## Dataset Prep
Similar to the other tutorials, the bace dataset will be used.

In [2]:
data = pd.read_csv("../dataset/bace.csv")
cols = ['CID', 'smiles', 'Class', 'Model']
dataset = data[cols]
print(f"The shape of the dataset: {dataset.shape}")
dataset.head()

The shape of the dataset: (1513, 4)


,CID,smiles,Class,Model
0,BACE_1,O1CC[C@@H](NC(=O)[C@@H](Cc2cc3cc(ccc3nc2N)-c2c...,1,train
1,BACE_2,Fc1cc(cc(F)c1)C[C@H](NC(=O)[C@@H](N1CC[C@](NC(...,1,train
2,BACE_3,S1(=O)(=O)N(c2cc(cc3c2n(cc3CC)CC1)C(=O)N[C@H](...,1,train
3,BACE_4,S1(=O)(=O)C[C@@H](Cc2cc(O[C@H](COCC)C(F)(F)F)c...,1,train
4,BACE_5,S1(=O)(=O)N(c2cc(cc3c2n(cc3CC)CC1)C(=O)N[C@H](...,1,train


The dataset is separated by train/test labels. The molecule smiles string and class label will be extracted from the table and, in a list, will be converted into molecular graph data. The first five entries in train_graphs will be printed to quickly visualize the data.

In [3]:
train = dataset[dataset.iloc[:, -1] == 'train']
test = dataset[dataset.iloc[:, -1] == 'test']

train_graphs = [MolToGraph(smi=smi, label=y).graph_data for smi, y in tqdm(zip(train.smiles, train.Class), desc="Converting Train Graphs")]
test_graphs = [MolToGraph(smi=smi, label=y).graph_data for smi, y in tqdm(zip(test.smiles, test.Class), desc="Converting Test Graphs")]

train_graphs[:5]

Converting Train Graphs: 1376it [00:02, 633.21it/s]
Converting Test Graphs: 137it [00:00, 651.99it/s]


[Data(x=[32, 142], edge_index=[2, 70], edge_attr=[70, 16], y=[1]),
 Data(x=[47, 142], edge_index=[2, 100], edge_attr=[100, 16], y=[1]),
 Data(x=[42, 142], edge_index=[2, 92], edge_attr=[92, 16], y=[1]),
 Data(x=[40, 142], edge_index=[2, 84], edge_attr=[84, 16], y=[1]),
 Data(x=[44, 142], edge_index=[2, 96], edge_attr=[96, 16], y=[1])]

## GCN
In this example, the GCN model will be used. By dfault, it will utilize AtomGraph features. Printing the model will show the graph model structure. In this example, three graph layers are used with an output of 128 dimensions.

In [4]:
graph_model = GCN(node_dim=142, hidden_dim=128, graph_layers=3, reduce_graph=False)
graph_model

GCN(
  (gcn_conv): ModuleList(
    (0): GCNConv(142, 128)
    (1-2): 2 x GCNConv(128, 128)
  )
  (normalization): ModuleList(
    (0-2): 3 x BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  )
)

### Extract Train/Test Data

Once the graph model has been created, the graph latent space can be extracted. This is done using the **split_graph_space** and will return train_x, train_y, test_x, and test_y data. This is helpful as it can be used to feed data directly into sklearn models.

In [5]:
train_x, train_y, test_x, test_y = graph_model.split_graph_space(train_graphs, test_graphs)

Converting test data: 100%|██████████| 137/137 [00:00<00:00, 1334.50it/s]


Visualizing the latent space vector gives the graph latent space as a vector matching the final hidden dimensions. In this case, the vector for the training data is used.

In [6]:
train_x

array([[0.02439971, 0.        , 0.        , ..., 0.        , 0.11893181,
        0.        ],
       [0.02434067, 0.        , 0.        , ..., 0.        , 0.1159888 ,
        0.        ],
       [0.03232265, 0.        , 0.0001815 , ..., 0.        , 0.11242544,
        0.        ],
       ...,
       [0.02210586, 0.        , 0.00198167, ..., 0.        , 0.13871397,
        0.        ],
       [0.03222104, 0.        , 0.        , ..., 0.        , 0.13180393,
        0.        ],
       [0.03701699, 0.        , 0.        , ..., 0.        , 0.12573563,
        0.        ]], shape=(1376, 128), dtype=float32)

## Random Forest Model

Once the graph latent space is generated, the arrays can be used to fit an sklearn model. In this case, a Random Forest Classifier.

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(train_x, train_y)

y_pred = rf.predict(test_x)
print("Accuracy:", accuracy_score(test_y, y_pred))

Accuracy: 0.8029197080291971


## Making Predictions

Making predictions requires a similar workflow - converting smiles string into graph data, obtaining the graph latent space, and using the fitted sklearn model for prediction.

**NOTE:** The molecules used for this example were taken from the training dataset above. Thus, the model has "cheated" and seen this data before. Future testing should be done using unseen data. 

In [8]:
# prediction
to_predict = ["O1CC[C@@H](NC(=O)[C@@H](Cc2cc3cc(ccc3nc2N)-c2ccccc2C)C)CC1(C)C", "Fc1cc(cc(F)c1)C[C@H](NC(=O)[C@@H](N1CC[C@](NC(=O)C)(CC(C)C)C1=O)CCc1ccccc1)[C@H](O)[C@@H]1[NH2+]C[C@H](OCCC)C1", "FC(F)(F)Oc1ccc(cc1)C1(N=C(N)N(C)C1=O)c1cc(ccc1)-c1ccc(OC)nc1"]

data = [MolToGraph(smi=smi).graph_data for smi in tqdm(to_predict, desc="Converting Data to Graphs")]

graph_latent_space = graph_model.extract_graph_space(data)
graph_latent_space

Converting data: 100%|██████████| 1/1 [00:00<00:00, 357.51it/s]


array([[3.10870465e-02, 0.00000000e+00, 0.00000000e+00, 1.16393147e-02,
        0.00000000e+00, 0.00000000e+00, 1.70242635e-03, 1.85160622e-01,
        0.00000000e+00, 8.36238787e-02, 1.30219785e-02, 0.00000000e+00,
        7.31484848e-04, 7.54388701e-03, 2.09940113e-02, 0.00000000e+00,
        5.50461933e-02, 4.73571531e-02, 3.20305303e-03, 6.88435510e-04,
        1.00465946e-01, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        1.51389092e-01, 0.00000000e+00, 2.61819532e-05, 1.31887314e-03,
        7.68355131e-02, 1.35830138e-02, 1.18724100e-04, 2.84306402e-03,
        2.53214804e-03, 0.00000000e+00, 0.00000000e+00, 9.24329460e-02,
        0.00000000e+00, 3.01244878e-03, 1.33699840e-02, 6.34191465e-03,
        3.85319255e-02, 1.45998076e-01, 1.23894915e-01, 2.10624561e-02,
        0.00000000e+00, 1.28911240e-02, 0.00000000e+00, 9.15136933e-02,
        0.00000000e+00, 2.43471540e-03, 5.73768616e-02, 1.76553652e-02,
        3.26074846e-02, 0.00000000e+00, 9.66032222e-03, 6.869938

### Predictions
With the molecular graph features generated, the features can be given to the **.predict** method in the Random Forest Classifier. This will output an array containing the binary classificaiton for the molecules.

In [9]:
predictions = rf.predict(graph_latent_space)
predictions

array([1, 1, 1])

Optionally, users can also output the results as probability.

In [10]:
predictions = rf.predict_proba(graph_latent_space)
predictions

array([[0.23, 0.77],
       [0.13, 0.87],
       [0.46, 0.54]])